# Notebook 03 — Consistency Score Formula (#83)

**Objective:** Design and validate club vs national performance scores for dual-context players.

**Inputs (Databricks → BigQuery foreign catalog):**
- `int_player_club_vs_national` — per-90 metrics and context z-scores (270+ min per context, both contexts required)
- `pca_loadings` — PCA feature weights from Quan's clustering pipeline

**Outputs:**
- Documented formula and weight rationale
- Feature weight table, scores table, quadrant plot, top 10 per quadrant
- **§8:** `consistency_scores.parquet` → GCS + `analytics.consistency_scores` in BigQuery

**Follow-up:** Join into `mart_player_performance` via `sources.yml`

**Research question:** Do players perform differently for club vs national teams? (RQ3)

---
## 0. Setup

In [0]:
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid", palette="muted")

# Databricks foreign catalog → BigQuery (see docs/databricks_bigquery_setup.md)
BQ_CATALOG = os.environ.get("BQ_CATALOG", "bq_raw_statsbomb_sa_catalog")
BQ_PCA_LOADINGS = f"{BQ_CATALOG}.analytics.pca_loadings"
# dbt writes intermediate models to raw_statsbomb_intermediate (not dbt_intermediate)
INT_TABLE = f"{BQ_CATALOG}.raw_statsbomb_intermediate.int_player_club_vs_national"

print(f"Players table: {INT_TABLE}")
print(f"PCA loadings:  {BQ_PCA_LOADINGS}")
print("Setup complete.")

---
## 1. Feature set (aligned with clustering / PCA)

Same 11 features as `player_clustering.ipynb` and `src/ml/cluster.py`.  
Z-scores are **pre-computed in dbt** per context (`is_international`): club players vs club population, national vs national population.

In [0]:
FEATURES = [
    "shots_per_90",
    "xg_per_90",
    "xg_per_shot",
    "dribbles_per_90",
    "carries_att_third_per_90",
    "passes_att_third_per_90",
    "pass_completion_pct",
    "pressures_per_90",
    "interceptions_per_90",
    "clearances_per_90",
    "aerial_duels_per_90",
]

Z_COLS = [f"z_{f}" for f in FEATURES]

print(f"{len(FEATURES)} PCA-aligned features")

---
## 2. Formula (documented)

### 2.1 Feature weights from PCA loadings

For each feature \(f\), sum absolute loadings across all principal components, then normalize:

$$
w_f = \frac{\sum_i |\text{loading}_{i,f}|}{\sum_{f'} \sum_i |\text{loading}_{i,f'}|}
$$

**Rationale:** Features that load strongly on more PCs (in absolute terms) contribute more to the overall playing-style space used for clustering. Normalizing makes weights interpretable and sum to 1.

*Alternative (not used here):* PC1-only weights — simpler but ignores secondary dimensions of style.

### 2.2 Context performance scores

For each player in one context (club or national), using dbt z-scores \(z_{f}\):

$$
\text{performance\_score} = \sum_f z_f \cdot w_f
$$

- `club_performance_score` — row where `is_international = false`
- `national_performance_score` — row where `is_international = true`

Higher score = stronger profile **relative to peers in that same context**.

### 2.3 Consistency score (club vs national similarity)

$$
\text{consistency\_score} = 1 - \frac{1}{|F|} \sum_f \left| z_{f,\text{club}} - z_{f,\text{national}} \right|
$$

**Rationale:** Mean absolute z-gap across the same 11 features. Bounded roughly in \([-2, 2]\) per feature gap → score near 1 means similar tactical profile across contexts; lower means larger club/national shift.

### 2.4 Performance quadrants

Median splits on `club_performance_score` and `national_performance_score`:

| Club | National | Label |
|------|----------|-------|
| High | High | Elite both |
| High | Low | Club-strong |
| Low | High | National-strong |
| Low | Low | Lower both |

---
## 3. Load data

In [0]:
players = spark.sql(f"SELECT * FROM {INT_TABLE}").toPandas()
loadings = spark.sql(
    f"SELECT component, feature, loading FROM {BQ_PCA_LOADINGS}"
).toPandas()


def normalize_is_international(val):
    """Spark/BQ may return bool, int, or string — map to True=national, False=club."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan
    if isinstance(val, (bool, np.bool_)):
        return bool(val)
    if isinstance(val, (int, float)):
        return bool(val)
    s = str(val).strip().lower()
    if s in ("true", "1", "t", "yes", "national"):
        return True
    if s in ("false", "0", "f", "no", "club"):
        return False
    raise ValueError(f"Unrecognized is_international value: {val!r}")


players["is_international"] = players["is_international"].apply(normalize_is_international)
players = players.dropna(subset=["is_international"]).copy()
players["is_international"] = players["is_international"].astype(bool)

n_players = players["player_id"].nunique()
print(f"Rows loaded: {len(players):,}")
print(f"Distinct players: {n_players:,}")
print(f"is_international counts:\n{players['is_international'].value_counts()}")
print(f"Loadings: {loadings['feature'].nunique()} features, {loadings['component'].nunique()} PCs")

# Flag duplicate player+context rows (e.g. name variants) before §5
dup_mask = players.duplicated(subset=["player_id", "is_international"], keep=False)
if dup_mask.any():
    print(f"Note: {dup_mask.sum()} duplicate player+context rows — will collapse in §5")
    display(players.loc[dup_mask, ["player_id", "player_name", "is_international"]].head(10))

display(players.head(5))

---
## 4. Derive feature weights

In [0]:
missing_in_loadings = set(FEATURES) - set(loadings["feature"])
if missing_in_loadings:
    raise ValueError(f"pca_loadings missing features: {missing_in_loadings}")

weights = (
    loadings.groupby("feature")["loading"]
    .apply(lambda s: s.abs().sum())
    .reindex(FEATURES)
)
weights = weights / weights.sum()

weights_df = pd.DataFrame({"feature": weights.index, "weight": weights.values})
weights_df["weight_pct"] = (weights_df["weight"] * 100).round(2)
weights_df = weights_df.sort_values("weight", ascending=False).reset_index(drop=True)

print(f"Weights sum to: {weights.sum():.6f}")
display(weights_df)

---
## 5. Compute scores

In [0]:
w = weights.to_dict()


def weighted_performance(row: pd.Series) -> float:
    return float(np.nansum([row[f"z_{f}"] * w[f] for f in FEATURES]))


players = players.copy()
players["performance_score"] = players.apply(weighted_performance, axis=1)

# One row per player per context (handles duplicate name spellings from source data)
agg_cols = {
    "player_name": "first",
    "performance_score": "first",
    "total_minutes": "first",
    **{c: "first" for c in Z_COLS},
}
players = (
    players.groupby(["player_id", "is_international"], as_index=False)
    .agg(agg_cols)
)

club = (
    players.loc[~players["is_international"], [
        "player_id", "player_name", "performance_score", "total_minutes"
    ]]
    .rename(columns={
        "performance_score": "club_performance_score",
        "total_minutes": "club_minutes",
    })
)

national = (
    players.loc[players["is_international"], [
        "player_id", "performance_score", "total_minutes"
    ]]
    .rename(columns={
        "performance_score": "national_performance_score",
        "total_minutes": "national_minutes",
    })
)

scores = pd.merge(
    club,
    national,
    on="player_id",
    how="inner",
    validate="one_to_one",
)

# Consistency: 1 - mean|z_club - z_nat| (vectorized; avoids apply/type mismatch on Databricks)
club_z = players.loc[~players["is_international"]].set_index("player_id")[Z_COLS]
nat_z = players.loc[players["is_international"]].set_index("player_id")[Z_COLS]
common_ids = club_z.index.intersection(nat_z.index)
z_gap = (club_z.loc[common_ids] - nat_z.loc[common_ids]).abs().mean(axis=1)
scores["consistency_score"] = scores["player_id"].map(1.0 - z_gap)

if len(scores) != n_players:
    print(
        f"Rows after merge: {len(scores):,} vs distinct players: {n_players:,}. "
        "Some players may be missing a context row."
    )

print(f"Scored players: {len(scores):,}")
print(f"Non-null consistency scores: {scores['consistency_score'].notna().sum():,}")
display(scores.head(10))
display(scores[[
    "club_performance_score",
    "national_performance_score",
    "consistency_score",
]].describe())

In [0]:
club_med = scores["club_performance_score"].median()
nat_med = scores["national_performance_score"].median()


def assign_quadrant(row: pd.Series) -> str:
    high_club = row.club_performance_score >= club_med
    high_nat = row.national_performance_score >= nat_med
    if high_club and high_nat:
        return "Elite both"
    if high_club and not high_nat:
        return "Club-strong"
    if not high_club and high_nat:
        return "National-strong"
    return "Lower both"


scores["performance_quadrant"] = scores.apply(assign_quadrant, axis=1)
print("Quadrant counts:")
display(scores["performance_quadrant"].value_counts().to_frame("count"))

# Stash for §8 export (re-run export without recomputing scores in the same session)
CONSISTENCY_EXPORT_VIEW = "_consistency_scores_export"
spark.createDataFrame(scores).createOrReplaceTempView(CONSISTENCY_EXPORT_VIEW)
print(f"Stashed {len(scores):,} rows → temp view `{CONSISTENCY_EXPORT_VIEW}`")

---
## 6. Quadrant scatter plot

In [0]:
fig, ax = plt.subplots(figsize=(9, 7))

palette = {
    "Elite both": "#2ca02c",
    "Club-strong": "#1f77b4",
    "National-strong": "#ff7f0e",
    "Lower both": "#7f7f7f",
}

for label, grp in scores.groupby("performance_quadrant"):
    ax.scatter(
        grp["club_performance_score"],
        grp["national_performance_score"],
        label=label,
        alpha=0.65,
        s=40,
        c=palette.get(label, "#333333"),
    )

ax.axvline(club_med, color="black", ls="--", lw=0.8, alpha=0.5)
ax.axhline(nat_med, color="black", ls="--", lw=0.8, alpha=0.5)
ax.set_xlabel("Club performance score (PCA-weighted z-sum)")
ax.set_ylabel("National performance score (PCA-weighted z-sum)")
ax.set_title(f"Club vs national performance ({len(scores):,} dual-context players)")
ax.legend(title="Quadrant", loc="best")
plt.tight_layout()
plt.show()

In [0]:
def top_per_quadrant(n: int = 10) -> pd.DataFrame:
    rows = []
    for q in scores["performance_quadrant"].unique():
        sub = scores.loc[scores["performance_quadrant"] == q].copy()
        sub["avg_performance"] = (
            sub["club_performance_score"] + sub["national_performance_score"]
        ) / 2
        top = sub.nlargest(n, "avg_performance")[
            [
                "player_name",
                "performance_quadrant",
                "club_performance_score",
                "national_performance_score",
                "consistency_score",
                "club_minutes",
                "national_minutes",
            ]
        ]
        rows.append(top)
    return pd.concat(rows, ignore_index=True)


display(top_per_quadrant(10))

---
## 7. Full results (display)

In [0]:
result_cols = [
    "player_id",
    "player_name",
    "club_performance_score",
    "national_performance_score",
    "consistency_score",
    "performance_quadrant",
    "club_minutes",
    "national_minutes",
]

print("Players per quadrant:")
display(scores["performance_quadrant"].value_counts().to_frame("count"))

print("\nAll dual-context player scores:")
display(scores[result_cols].sort_values("player_name"))

---
## 8. Export to GCS and BigQuery (#84)

Run after **§3–§6** (must define `scores` and quadrants). Uses a **service account JSON** for the Python BigQuery/GCS client (separate from the Lakehouse SQL connection).

**Outputs:**
- `gs://{bucket}/models/consistency/consistency_scores.parquet`
- `{PROJECT_ID}.analytics.consistency_scores`

In [0]:
import json
import os
import subprocess
import sys
import tempfile

# Requires `scores` from §5–§6 (quadrants). If you only re-run this cell, load from the temp view.
CONSISTENCY_EXPORT_VIEW = "_consistency_scores_export"
if "scores" not in globals():
    try:
        scores = spark.table(CONSISTENCY_EXPORT_VIEW).toPandas()
        print(f"Loaded `scores` from temp view `{CONSISTENCY_EXPORT_VIEW}` ({len(scores):,} rows)")
    except Exception as exc:
        raise RuntimeError(
            "Variable `scores` is not defined. Run §3 through §6 (quadrant cell) in this session, "
            "then re-run this export cell."
        ) from exc

PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "football-capstone-mds-496219")
DATASET = os.environ.get("ML_BQ_DATASET", "analytics")
TABLE = "consistency_scores"
GCS_BUCKET = os.environ.get("ML_GCS_BUCKET", "football-analytics-mds496219")
GCS_OBJECT = "models/consistency/consistency_scores.parquet"

result_cols = [
    "player_id",
    "player_name",
    "club_performance_score",
    "national_performance_score",
    "consistency_score",
    "performance_quadrant",
    "club_minutes",
    "national_minutes",
]

export_df = scores[result_cols].copy()
export_df["player_id"] = export_df["player_id"].astype("int64")

# Optional: Spark DataFrame wrapper (for Option A / C below)
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()
# out = spark.createDataFrame(export_df)

# --- Option A — Spark → BigQuery (classic / pro clusters with spark-bigquery JAR only; NOT Serverless) ---
# out.write.format("bigquery").mode("overwrite").option("writeMethod", "direct").option(
#     "table", f"{PROJECT_ID}.{DATASET}.{TABLE}"
# ).save()

# --- Option B — pandas → GCS parquet + BigQuery (works on Serverless with SA JSON) ---
BQ_SERVICE_ACCOUNT_JSON_PATH = "/Volumes/workspace/default/gcp_keys/bq-sa.json"
BQ_SECRET_SCOPE = None  # e.g. "capstone-gcp"
BQ_SECRET_KEY = None  # e.g. "bq_service_account_json"


def _gcp_credentials():
    json_path = (
        os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
        or os.environ.get("BQ_SERVICE_ACCOUNT_JSON_PATH")
        or BQ_SERVICE_ACCOUNT_JSON_PATH
    )
    if json_path and os.path.isfile(json_path):
        from google.oauth2 import service_account

        return service_account.Credentials.from_service_account_file(json_path)
    if BQ_SECRET_SCOPE and BQ_SECRET_KEY:
        _dbu = globals().get("dbutils")
        if _dbu is None:
            try:
                from databricks.sdk.runtime import dbutils as _dbu
            except ImportError:
                _dbu = None
        if _dbu is None:
            raise RuntimeError(
                "BQ_SECRET_SCOPE/BQ_SECRET_KEY are set but dbutils is not available."
            )
        from google.oauth2 import service_account

        info = json.loads(_dbu.secrets.get(scope=BQ_SECRET_SCOPE, key=BQ_SECRET_KEY))
        return service_account.Credentials.from_service_account_info(info)
    return None


def _ensure_google_cloud():
    for pkg in ("google-cloud-bigquery", "google-cloud-storage"):
        try:
            if pkg == "google-cloud-bigquery":
                from google.cloud import bigquery  # noqa: F401
            else:
                from google.cloud import storage  # noqa: F401
        except ImportError:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", pkg],
                stdout=subprocess.DEVNULL,
            )
            for _name in ("google.cloud.bigquery", "google.cloud.storage", "google.cloud", "google"):
                sys.modules.pop(_name, None)


_ensure_google_cloud()
from google.auth.exceptions import DefaultCredentialsError
from google.cloud import bigquery, storage

creds = _gcp_credentials()
if creds is None:
    try:
        bq_client = bigquery.Client(project=PROJECT_ID)
        storage_client = storage.Client(project=PROJECT_ID)
    except DefaultCredentialsError as exc:
        raise RuntimeError(
            "No GCP credentials for the Python client. Set GOOGLE_APPLICATION_CREDENTIALS, "
            "BQ_SERVICE_ACCOUNT_JSON_PATH, or BQ_SECRET_SCOPE/BQ_SECRET_KEY. "
            "The Lakehouse BQ SQL connection does not provide ADC for this API."
        ) from exc
else:
    bq_client = bigquery.Client(project=PROJECT_ID, credentials=creds)
    storage_client = storage.Client(project=PROJECT_ID, credentials=creds)

with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
    local_parquet = tmp.name
export_df.to_parquet(local_parquet, index=False)

gcs_uri = f"gs://{GCS_BUCKET}/{GCS_OBJECT}"
bucket = storage_client.bucket(GCS_BUCKET)
bucket.blob(GCS_OBJECT).upload_from_filename(local_parquet)
os.unlink(local_parquet)
print(f"Uploaded {len(export_df):,} rows → {gcs_uri}")

table_id = f"{PROJECT_ID}.{DATASET}.{TABLE}"
load_job = bq_client.load_table_from_dataframe(
    export_df,
    table_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
load_job.result()
print(f"Loaded {load_job.output_rows:,} rows → {table_id}")

display(export_df.head(5))

# --- Option C — Delta in Unity Catalog (Serverless-friendly) ---
# out.write.mode("overwrite").format("delta").saveAsTable("main.analytics.consistency_scores")

---
## 9. Summary

| Item | Value |
|------|-------|
| Dual-context players | See §3 load cell |
| Export table | `analytics.consistency_scores` |
| GCS path | `models/consistency/consistency_scores.parquet` |
| Features | 11 (PCA-aligned) |
| Weights | Σ\|loading\| across PCs, normalized |
| Club / national score | Weighted sum of context z-scores |
| Consistency | 1 − mean \|Δz\| across features |
| Quadrants | Median splits on club vs national scores |

### Follow-up
- Add `consistency_scores` to `models/sources.yml` and join into `mart_player_performance`